In [6]:
import sys
sys.path.append('..')

import os
from sqlalchemy import create_engine, text
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon, MultiPoint
from shapely import wkb
from geopandas.tools import sjoin

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

RUTA_COMPLETA = os.path.join(RUTA_UNIDAD_ONE_DRIVE, RUTA_LOCAL_ONE_DRIVE)
#PATH_CAT = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Pulverizacion\2025\SHP\catastro_S09_MIERCOLES.shp'
PATH_CAT = ''
PATH_XLSX_GRUPOS = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2025\DATA\GRUPO_COSECHA.xlsx'

In [7]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_planificacion():
    engine = obtener_engine()
    try:
        query = """
            SELECT *
            FROM drones_pulverizacion.planificacion_pulv
            WHERE procesado=false;
        """
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"Error en la consulta: {e}")
        return gpd.GeoDataFrame()
    return None

def marcar_como_procesado(id_os):
    engine = obtener_engine()  # tu función que crea el engine
    try:
        with engine.begin() as conn:
            query = text("""
                UPDATE drones_pulverizacion.planificacion_pulv
                SET procesado = true
                WHERE id = :id_os
            """)
            conn.execute(query, {"id_os": id_os})
            print(f"✔️ id {id_os} marcado como procesado.")
    except Exception as e:
        print(f"❌ Error al actualizar: {e}")
    return None

def marcar_lote_como_planificado(id_lote):
    engine = obtener_engine()  # tu función que crea el engine
    try:
        with engine.begin() as conn:
            query = text("""
                UPDATE drones_pulverizacion.parte_diario_pulv SET estado = 'PLANIFICADO' WHERE id = :id_lote
            """)
            conn.execute(query, {"id_lote": id_lote})
            print(f"✔️ lote id={id_lote} planificado.")
    except Exception as e:
        print(f"❌ Error al actualizar: {e}")
    return None

def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

def get_catastro_adm():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro_adm
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

In [8]:
# obtiene el catastro agricola actual
gdf_cat = get_catastro()
# obtiene el catastro agricola actual
gdf_cat_adm = get_catastro_adm()
# obtiene los puntos planificados
gdf_plan = obtener_planificacion()
gdf_plan

,id,geom,codigo_canero,nombre_canero,procesado,mezcla,obs,fecha_registro
0,139,MULTIPOINT (489161.480 8103904.902),10697,PAZ REA JAIME EDUARDO,False,100,MADURADOR,2026-03-27 10:31:50.438769


# CARGAR PLANIFICACION A PARTE DIARIO

In [9]:
# lista de todos los ids nuevos en planificacion
ids_planificacion = list(set(gdf_plan['id']))
print(len(ids_planificacion), 'ordenes de servicio para procesar')

1 ordenes de servicio para procesar


In [10]:
# recorrer la lista de ids, seleccion uno de los registros de planificacion y los procesa para registrarlo en parte diario
for i in ids_planificacion:
    print(f"Procesando id: {i}")
    # filtra un registro de planificacion segun id
    plan = gdf_plan[gdf_plan['id'] == i]
    # intersecta registro con catastro_plan
    gdf_intersect_result = gpd.sjoin(gdf_cat_adm, plan, how='inner')
    if len(gdf_intersect_result) == 0:
        gdf_intersect_result = gpd.sjoin(gdf_cat, plan, how='inner')
    if len(gdf_intersect_result) == 0:
        continue
    # selecciona la columnas de interes
    gdf_result = gdf_intersect_result[['geom', 'unidad_01', 'unidad_02', 'codigo_canero', 'nombre_canero', 'unidad_05', 'area', 'id_right', 'soca', 'zona', 'mezcla']].copy()
    # recalcula en area
    gdf_result['area'] = gdf_result.geometry.area / 10000
    # renombra las columnas
    gdf_result.rename(columns={
        'geometry': 'geom',
        'codigo_canero': 'unidad_03',
        'nombre_canero': 'unidad_04',
        'id_right': 'os',
        'zona': 'inst'
    }, inplace=True)
    # asigna la columna de geometria
    gdf_result = gdf_result.set_geometry("geom")
    # cambia tipos de datos de columnas
    gdf_result['unidad_01'] = gdf_result['unidad_01'].astype(int)
    gdf_result['unidad_03'] = gdf_result['unidad_03'].astype(int)
    gdf_result['os'] = gdf_result['os'].astype(int)
    gdf_result['soca'] = gdf_result['soca'].astype(int)
    gdf_result['inst'] = gdf_result['inst'].astype(int)
    gdf_result['mezcla'] = gdf_result['mezcla'].astype(int)
    
    # GARANTIZAR LA INST DEL CAÑERO
    # cargar grupos de cosecha
    df_grupos = pd.read_excel(PATH_XLSX_GRUPOS, sheet_name='CODIGOS')
    df_grupos = df_grupos[df_grupos['INSTITUCION'].notna()]
    df_grupos['CODIGO CAÑERO'] = df_grupos['CODIGO CAÑERO'].astype(int)
    df_grupos['INSTITUCION'] = df_grupos['INSTITUCION'].astype(int)
    # Crear un diccionario de mapeo: {codigo_cañero: institucion}
    mapa_institucion = df_grupos.set_index('CODIGO CAÑERO')['INSTITUCION'].to_dict()
    # Reemplazar los valores de 'inst' en gdf_result usando el diccionario
    # a partir del diccionario, busca el codigo cañero (unidad_03), y reempalza la institucion
    gdf_result['inst'] = gdf_result['unidad_03'].map(mapa_institucion)
    
    # agregar nuevos registros a la base de datos
    gdf_result.to_postgis("parte_diario_pulv", obtener_engine(), schema="drones_pulverizacion", if_exists="append")
    marcar_como_procesado(i)

Procesando id: 139
✔️ id 139 marcado como procesado.
